<a href="https://colab.research.google.com/github/Corerishi/ML_2547243/blob/Lab3/Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import pickle

df = pd.read_csv("Student Awareness Survey.csv")

print("First 5 rows of the dataset:")
print(df.head())

print(f"\nDataset Dimensions: {df.shape}")

print("\nMissing Values Before Cleaning:")
print(df.isnull().sum())

df.drop_duplicates(inplace=True)

cols_needed = [
    'Your CIA % of last semester',
    'Your maximum attendance % till last semester',
    'Your GPA of last semester'
]
df_clean = df[cols_needed].copy()

df_clean.dropna(inplace=True)

df_clean = df_clean.apply(pd.to_numeric, errors='coerce')
df_clean.dropna(inplace=True)

print("\nStatistical Summary:")
print(df_clean.describe())

X_cia = df_clean[['Your CIA % of last semester']]
Y = df_clean['Your GPA of last semester']

X_att = df_clean[['Your maximum attendance % till last semester']]

First 5 rows of the dataset:
           Timestamp  Registration Number  \
0  6/15/2026 9:25:39              2547231   
1  6/15/2026 9:53:54              2547237   
2  6/15/2026 9:54:56              2547203   
3  6/15/2026 9:55:17              2547228   
4  6/15/2026 9:55:42              2547241   

                                        Email  \
0       kunnal.kunnal@mca.christuniversity.in   
1  omkaar.chakraborty@mca.christuniversity.in   
2        abhinav.jain@mca.christuniversity.in   
3          jai.pareek@mca.christuniversity.in   
4             r.karan@mca.christuniversity.in   

   Job role that you are interested in  \
0  Software Development Engineer (SDE)   
1  Software Development Engineer (SDE)   
2                 Full Stack Developer   
3                 Full Stack Developer   
4  Software Development Engineer (SDE)   

  What is the minimum salary of students placed through campus (In LPA..respond as a number)  \
0                                                3.5    

In [ ]:
def run_regression_comparison(X_df, Y_series, experiment_name):
    print(f"\n {experiment_name} ")
    X_train, X_test, y_train, y_test = train_test_split(X_df, Y_series, test_size=0.2, random_state=42)

    model = LinearRegression()
    model.fit(X_train, y_train)
    m_skl = model.coef_[0]
    b_skl = model.intercept_
    y_pred_skl = model.predict(X_test)

    x_train_vals = X_train.iloc[:, 0].values
    y_train_vals = y_train.values
    x_test_vals = X_test.iloc[:, 0].values

    x_mean = np.mean(x_train_vals)
    y_mean = np.mean(y_train_vals)

    numerator = np.sum((x_train_vals - x_mean) * (y_train_vals - y_mean))
    denominator = np.sum((x_train_vals - x_mean) ** 2)

    m_manual = numerator / denominator
    b_manual = y_mean - (m_manual * x_mean)

    y_pred_manual = (m_manual * x_test_vals) + b_manual

    print(f"Scikit-learn  -> Slope: {m_skl:.6f}, Intercept: {b_skl:.6f}")
    print(f"Manual OLS    -> Slope: {m_manual:.6f}, Intercept: {b_manual:.6f}")

    diff = np.mean(np.abs(y_pred_skl - y_pred_manual))
    print(f"Mean Difference in Predictions: {diff:.10f}")

    return m_skl, b_skl

slope_cia, intercept_cia = run_regression_comparison(X_cia, Y, "Experiment 1: CIA vs GPA")

slope_att, intercept_att = run_regression_comparison(X_att, Y, "Experiment 2: Attendance vs GPA")


 Experiment 1: CIA vs GPA 
Scikit-learn  -> Slope: 0.016526, Intercept: 2.212167
Manual OLS    -> Slope: 0.016526, Intercept: 2.212167
Mean Difference in Predictions: 0.0000000000

 Experiment 2: Attendance vs GPA 
Scikit-learn  -> Slope: 0.014862, Intercept: 2.012838
Manual OLS    -> Slope: 0.014862, Intercept: 2.012838
Mean Difference in Predictions: 0.0000000000


In [ ]:
params_to_save = {
    'slope': slope_cia,
    'intercept': intercept_cia
}

with open('linear_regression_weights.pkl', 'wb') as f:
    pickle.dump(params_to_save, f)
print("\nParameters saved to 'linear_regression_weights.pkl'")

with open('linear_regression_weights.pkl', 'rb') as f:
    loaded_params = pickle.load(f)

sample_cia = 85
predicted_gpa = (loaded_params['slope'] * sample_cia) + loaded_params['intercept']
print(f"Loaded Params: {loaded_params}")
print(f"Predicted GPA for a student with 85% CIA: {predicted_gpa:.2f}")


Parameters saved to 'linear_regression_weights.pkl'
Loaded Params: {'slope': np.float64(0.01652616755411487), 'intercept': np.float64(2.2121674902659123)}
Predicted GPA for a student with 85% CIA: 3.62
